In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from scipy.stats import uniform, loguniform, norm
from sklearn.gaussian_process.kernels import Matern, RBF, WhiteKernel, ConstantKernel as C
from scipy.optimize import minimize
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score

## **Function 8** - maximizing 8D black-box function

- Here we are optimising an eight-dimensional black-box function, where each of the eight input parameters affects the output, but the internal mechanics are unknown.

- **Objective** - find the parameter combination that maximises the function’s output, such as _performance_, _efficiency_ or _validation accuracy_.
  - Because the function is high-dimensional and likely complex, global optimisation is hard, so **identifying strong local maxima** is often a practical strategy.

- **For example**, imagine you’re tuning an ML model with eight hyperparameters:
  - learning rate
  - batch size
  - number of layers
  - dropout rate
  - regularisation strength
  - activation function (numerically encoded)
  - optimiser type (encoded)
  - initial weight range.

Each input set returns a single validation accuracy score between 0 and 1.

- **Goal** - maximise this score

In [2]:
# Load current cumulative dataset (original + prior weekly updates) for Function 8
X = np.load('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_8/initial_inputs.npy')
Y = np.load('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_8/initial_outputs.npy')

# New data for Week 6 (Function 8)
X_w5_new_point = np.array([0.029788, 0.370437, 0.009211, 0.563220, 0.914458, 0.813905, 0.149129, 0.554138], dtype=np.float64)
Y_w5_new_point = np.array([9.6165546476866], dtype=np.float64)

# Append the new data point
X_updated = np.vstack((X, X_w5_new_point.reshape(1, -1)))

# Remove duplicate X rows (if point already exists)
X_unique, unique_indices = np.unique(X_updated, axis=0, return_index=True)

# Build Y and keep matching rows only
Y_all = np.append(Y, Y_w5_new_point)
Y_updated = Y_all[unique_indices]

# Save updated arrays
np.save('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_8/initial_inputs.npy', X_unique)
np.save('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_8/initial_outputs.npy', Y_updated)

# Show updated arrays
print("Updated Inputs (X) - Function 8: ", X_unique)
print("Updated Outputs (Y) - Function 8: ", Y_updated)

Updated Inputs (X) - Function 8:  [[0.007838   0.337307   0.164679   0.407388   0.632409   0.551169
  0.306982   0.5023    ]
 [0.00907698 0.81162615 0.52052036 0.07568668 0.26511183 0.09165169
  0.59241515 0.36732026]
 [0.02894663 0.02827906 0.48137155 0.6131746  0.67266045 0.02211341
  0.6014833  0.52488505]
 [0.029788   0.370437   0.009211   0.56322    0.914458   0.813905
  0.149129   0.554138  ]
 [0.0417     0.167136   0.185989   0.004377   0.961301   0.357272
  0.961301   0.357272  ]
 [0.04432925 0.01358149 0.25819824 0.57764416 0.05127992 0.15856307
  0.59103012 0.07795293]
 [0.05644741 0.06595555 0.02292868 0.03878647 0.40393544 0.80105533
  0.48830701 0.89308498]
 [0.110613   0.166799   0.129222   0.228937   0.996353   0.475272
  0.30197    0.476277  ]
 [0.11207131 0.43773566 0.59659878 0.59277563 0.22698177 0.41010452
  0.92123758 0.67475276]
 [0.117464   0.226763   0.160734   0.103849   0.998846   0.763666
  0.087753   0.474624  ]
 [0.1435503  0.93741452 0.23232482 0.00904349 

### **Output Analysis**
- Last week (W4) I saw $8.7639$, and this week (W5) I saw $9.6166$.

- **Real-World Implication**: This is a positive trend. 
    - In the context of a robot sensor, a higher value suggests a stronger signal-to-noise ratio or better calibration. 
    - I am moving toward a more reliable state for the robot's navigation or detection systems.

- **Model Clues**: Since this function is described as "stochastic" (noisy), the jump from 8.7 to 9.6 is encouraging, but I must be cautious. 
    - Because of the noise, a single high result can be a lucky reading rather than a true peak. 
    - However, two consecutive high results ($8.7$ and $9.6$) suggest I am genuinely centered in a region of high performance within the 8D space.

### **Bayesian Optimisation**: Gaussian Process Regressor

- **Model Understanding**: For an 8D problem, ARD (Automatic Relevance Determination) is not just helpful, it is mandatory. 
    - It allows the model to differentiate between critical sensor parameters and those that might just be adding noise. 
    - I use a Matern kernel (`nu=2.5`) to maintain a smooth surrogate surface.

- **Changes/Reasons**: I am maintaining the ARD structure and the WhiteKernel. 
    - The WhiteKernel is particularly vital here to prevent the GP from over-fitting to the stochastic noise. 
    - By keeping the model consistent with my W5 architecture, I allow the GP to aggregate these high results and build a more robust estimate of the true mean.

- **Intended Impact**: To use the 5 weeks of data to filter out the noise and identify which of the 8 dimensions are actually driving the increased sensor performance.

In [3]:
# Define the model
kernel = C(1.0) * Matern(length_scale=[0.1]*8, nu=2.5) + WhiteKernel(noise_level=1e-3)

model = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=25,
    normalize_y=True
)

# Fit the model
model.fit(X_unique, Y_updated)

/Users/prathamyeole/Library/Python/3.13/lib/python/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/prathamyeole/Library/Python/3.13/lib/python/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


,"kernel kernel: kernel instance, default=NoneThe kernel specifying the covariance function of the GP. If None ispassed, the kernel ``ConstantKernel(1.0, constant_value_bounds=""fixed"")* RBF(1.0, length_scale_bounds=""fixed"")`` is used as default. Note thatthe kernel hyperparameters are optimized during fitting unless thebounds are marked as ""fixed"".",1**2 * Matern...e_level=0.001)
,"alpha alpha: float or ndarray of shape (n_samples,), default=1e-10Value added to the diagonal of the kernel matrix during fitting.This can prevent a potential numerical issue during fitting, byensuring that the calculated values form a positive definite matrix.It can also be interpreted as the variance of additional Gaussianmeasurement noise on the training observations. Note that this isdifferent from using a `WhiteKernel`. If an array is passed, it musthave the same number of entries as the data used for fitting and isused as datapoint-dependent noise level. Allowing to specify thenoise level directly as a parameter is mainly for convenience andfor consistency with :class:`~sklearn.linear_model.Ridge`.For an example illustrating how the alpha parameter controlsthe noise variance in Gaussian Process Regression, see:ref:`sphx_glr_auto_examples_gaussian_process_plot_gpr_noisy_targets.py`.",1e-10
,"optimizer optimizer: ""fmin_l_bfgs_b"", callable or None, default=""fmin_l_bfgs_b""Can either be one of the internally supported optimizers for optimizingthe kernel's parameters, specified by a string, or an externallydefined optimizer passed as a callable. If a callable is passed, itmust have the signature:: def optimizer(obj_func, initial_theta, bounds): # * 'obj_func': the objective function to be minimized, which # takes the hyperparameters theta as a parameter and an # optional flag eval_gradient, which determines if the # gradient is returned additionally to the function value # * 'initial_theta': the initial value for theta, which can be # used by local optimizers # * 'bounds': the bounds on the values of theta .... # Returned are the best found hyperparameters theta and # the corresponding value of the target function. return theta_opt, func_minPer default, the L-BFGS-B algorithm from `scipy.optimize.minimize`is used. If None is passed, the kernel's parameters are kept fixed.Available internal optimizers are: `{'fmin_l_bfgs_b'}`.",'fmin_l_bfgs_b'
,"n_restarts_optimizer n_restarts_optimizer: int, default=0The number of restarts of the optimizer for finding the kernel'sparameters which maximize the log-marginal likelihood. The first runof the optimizer is performed from the kernel's initial parameters,the remaining ones (if any) from thetas sampled log-uniform randomlyfrom the space of allowed theta-values. If greater than 0, all boundsmust be finite. Note that `n_restarts_optimizer == 0` implies that onerun is performed.",25
,"normalize_y normalize_y: bool, default=FalseWhether or not to normalize the target values `y` by removing the meanand scaling to unit-variance. This is recommended for cases wherezero-mean, unit-variance priors are used. Note that, in thisimplementation, the normalisation is reversed before the GP predictionsare reported... versionchanged:: 0.23",True
,"copy_X_train copy_X_train: bool, default=TrueIf True, a persistent copy of the training data is stored in theobject. Otherwise, just a reference to the training data is stored,which might cause predictions to change if the data is modifiedexternally.",True
,"n_targets n_targets: int, default=NoneThe number of dimensions of the target values. Used to decide the numberof outputs when sampling from the prior distributions (i.e. calling:meth:`sample_y` before :meth:`fit`). This parameter is ignored once:meth:`fit` has been called... versionadded:: 1.3",None
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation used to initialize the centers.Pass an int for reproducible results across multiple function calls.See :term:`G

### **Acquisition Function**

- **Function Understanding**: I am using Upper Confidence Bound (UCB) with $\beta=2.0$.

- **Changes/Reasons**: I am staying with $\beta=2.0$. In an 8D space, the volume of unexplored territory is massive.
    - I cannot afford to purely exploit my $9.61$ result yet because the noise might be masking a $12.0+$ result just a few units away. 
        - $\beta=2.0$ keeps us exploring the uncertainty frontier while staying tethered to my current high-performing zone.

- **Intended Impact**: To navigate the 8D space by targeting regions that offer a high predicted mean but still possess enough uncertainty to potentially hide the true global maximum.

In [4]:
def upper_confidence_bound(X, model, beta=2.0):
    mu, sigma = model.predict(X, return_std=True)
    return mu + beta * sigma

# Generate an 8D random grid (200,000 points)
x_grid = np.random.uniform(0, 1, (200000, 8))

# Calculate UCB values using Beta=2.0
ucb_values = upper_confidence_bound(x_grid, model, beta=2.0)

# Select the next query
next_query = x_grid[np.argmax(ucb_values)]

print(f"Strategic Week 6 Query (Function 8): [{next_query[0]:.6f}-{next_query[1]:.6f}-{next_query[2]:.6f}-{next_query[3]:.6f}-{next_query[4]:.6f}-{next_query[5]:.6f}-{next_query[6]:.6f}-{next_query[7]:.6f}]")


Strategic Week 6 Query (Function 8): [0.258372-0.326752-0.066404-0.126235-0.943892-0.890864-0.283196-0.101725]
